# 03 — Layer 2: group construction and the non-circular signature anchor (RQ3)
Builds taxonomy-based groups from the registered (noisy) category and validates that the label-aware axis **C tracks the real per-group noise rate** (measured from clean labels). Because the noise is real (not injected), this is the non-circular anchor of §4.2(B). Also reports the $(C,\kappa,W,D)$ tuple distribution across groups.

In [2]:
# Notebook: 03_layer2_signature
# Shared plotting style: grayscale seaborn, dpi 600, PNG + PDF, no captions.
import os, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["axes.edgecolor"] = "0.2"; plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["font.family"] = "DejaVu Sans"
GREYS = ["#111111", "#555555", "#888888", "#bbbbbb", "#dddddd"]
FIG = os.path.join("..", "results", "figures"); TAB = os.path.join("..", "results", "tables")
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, f"{name}.{ext}"), dpi=600, bbox_inches="tight")
    plt.close(fig)

import sys; sys.path.append(os.path.join("..", "src"))
import numpy as np
from guard_core import guard_by_group
from scipy.stats import spearmanr
DATA_DIR = os.path.join("..", "data")
meta = pd.read_parquet(os.path.join(DATA_DIR, "item_meta.parquet"))
pi = np.load(os.path.join(DATA_DIR, "pi_memmap.npy"), mmap_mode="r")
N, K = pi.shape

# --- group by registered (noisy) category id; keep groups of a workable size ---
group_ids = meta["noisy_id"].values
a = meta["noisy_id"].values
sizes = pd.Series(group_ids).value_counts()
keep = set(sizes[(sizes >= 20) & (sizes <= 5000)].index)   # avoid tiny/huge groups
mask = np.isin(group_ids, list(keep))
print(f"groups kept: {len(keep):,}  items in kept groups: {mask.sum():,}")

rows = guard_by_group(pi, a[mask], group_ids[mask], K, min_size=20)
g = pd.DataFrame(rows)
# attach the real per-group noise rate from clean labels
valid = meta["clean_id"].values >= 0
mis = meta["is_misregistered"].values
noise_by_group = (pd.Series(mis[mask & valid])
                  .groupby(group_ids[mask & valid]).mean())
g["true_noise_rate"] = g["group"].map(noise_by_group)
g = g.dropna(subset=["true_noise_rate"])
g.to_csv(os.path.join(TAB, "t_layer2_groups.csv"), index=False)
print(g[["n", "C", "kappa", "W", "D", "true_noise_rate"]].describe().to_string())

# --- non-circular check: does C track the REAL group noise rate? ---
rho_C, p_C = spearmanr(g["C"], g["true_noise_rate"])
rho_Ck, p_Ck = spearmanr(g["C"] * g["kappa"], g["true_noise_rate"])
print(f"Spearman(C, true_noise_rate)      = {rho_C:.3f}  (p={p_C:.1e})")
print(f"Spearman(C*kappa, true_noise_rate)= {rho_Ck:.3f}  (p={p_Ck:.1e})")

fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.scatter(g["true_noise_rate"], g["C"], s=10, color=GREYS[1], edgecolor="none", alpha=0.5)
ax.set_xlabel("real per-group noise rate (from clean labels)")
ax.set_ylabel("C  (claim-belief JS)")
savefig(fig, "f_layer2_C_vs_noise")
print("saved f_layer2_C_vs_noise.{png,pdf}")

groups kept: 3,008  items in kept groups: 436,221
                 n            C        kappa            W            D  true_noise_rate
count  3008.000000  3008.000000  3008.000000  3008.000000  3008.000000      3008.000000
mean    145.020279     0.998608     0.572103     2.085385     1.613993         0.170982
std     308.979032     0.018146     0.167331     1.152010     0.474889         0.239487
min      20.000000     0.271761     0.132548     0.039908     0.474441         0.000000
25%      31.000000     0.999595     0.437089     1.105475     1.266233         0.008569
50%      54.000000     0.999861     0.583480     1.910016     1.584585         0.056395
75%     120.000000     0.999945     0.715391     2.948841     1.911799         0.229567
max    4124.000000     1.000035     0.926726     5.681867     3.673890         0.993590
Spearman(C, true_noise_rate)      = -0.060  (p=1.1e-03)
Spearman(C*kappa, true_noise_rate)= -0.095  (p=1.7e-07)
saved f_layer2_C_vs_noise.{png,pdf}
